# 🏆 SOTA-v30 6-모델 순차 분류 체인 & 하이브리드 블렌딩 파이프라인
이 주피터 노트북은 **DACON ETRI 멀티모달 라이프로그 기반 인간 이해 AI 논문 경진대회**를 위한 최신 SOTA-v30 모델링 파이프라인의 전체 소스 코드와 학술적 해설을 담고 있습니다.

### 🧬 핵심 설계 아키텍처 (Key Architectural Highlights):
1. **고성능 Baseline v9 피처 통합**: 수면 시간대 심박수(HR) 평균/표준편차/RMSSD, 수면 중 스파이크(Spike) 개수, 유저별 앱 사용 시간 엔트로피(Entropy), 조도 Dip 비율, 행동 세그먼트별 비율을 멀티모달 센서 Parquet 파일로부터 정밀 추출하여 피처스페이스를 비약적으로 보완했습니다.
2. **순차 분류 체인 (Sequential Classifier Chain)**: 수면/정서 지표 간의 인과적 연쇄 관계를 모방하기 위해, 예측 난이도가 상대적으로 쉬운 타겟부터 어려운 타겟 순서(`S3 -> S2 -> S1 -> Q3 -> Q2 -> S4 -> Q1`)로 확률 및 로짓 값을 예측하고 이를 다음 모델의 입력 피처로 누적 인젝션(OOF 방식)합니다.
3. **엄격한 시간 홀 교차 검증 (Subject-wise Hole CV)**: 동일 참가자의 미래 정보가 과거로 누수(Look-ahead leak)되는 것을 차단하기 위해, 참가자 단위로 데이터를 시간순 정렬한 뒤 중간에 구멍(Hole)을 파내는 방식의 10-Fold Block-CV를 수행하여 리더보드 테스트 셋과 100% 일치하는 환경을 제공합니다.
4. **Laplace-smoothed Target Encoding & Z-Scaler**: 주관적 정서 지표(Q1~Q3)가 참가자 개인 고유의 역사적 평균(Historical average) 대비 상대적 만족도를 뜻한다는 도메인 지식에 착안하여, 교차 검증 Fold 내부의 Train 데이터로만 Fit 되는 Z-score 스케일러와 Target Encoder를 구현해 오염을 완벽히 차단했습니다.
5. **안정적인 SLSQP 앙상블 블렌더**: 앙상블 가중치의 Global 합을 1.0으로 고정하고 boundary를 [0, 1]로 제한한 수치해석적 SLSQP Minimization 스태킹을 구현하여, gradient vanishing 없는 최적의 블렌더 확률 분포를 도출합니다.

## 🛠️ Part 1: 환경 세팅 및 라이브러리 로드

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

# 불필요한 경고 메시지를 억제합니다.
warnings.filterwarnings('ignore')

# 프로젝트 루트 디렉토리를 파이썬 경로에 추가하여 src 모듈을 임포트할 수 있게 합니다.
sys.path.append(str(Path.cwd()))
print("환경 세팅 완료!")

## 🧬 Part 2: 릭-프리(Leak-free) 검증 아키텍처 및 스케일러 구현
미래 정보 누수와 타겟 정보 오염(Target Leakage)을 완벽하게 차단하기 위해, 훈련 폴드(Train Fold) 내에서만 학습하여 평가 폴드(Validation/Test Fold)에 맵핑하는 사용자 정의 Z-score 스케일러, PCA, Laplace Target Encoder와 시계열 홀 교차 검증기(`make_subject_hole_folds`) 소스 코드 전체입니다.

In [ ]:
# ==============================================================================
# [스케일러 및 트랜스포머 아키텍처 전체 소스 코드]
# ==============================================================================

class LeakFreeSubjectZScaler(BaseEstimator, TransformerMixin):
    """
    A strictly leak-free subject-wise Z-Score Normalization transformer
    that fits mean and std statistics ONLY on the train folds
    and applies them to any input data without future look-ahead.
    """
    def __init__(self, id_col='subject_id', numeric_cols=None):
        self.id_col = id_col
        self.numeric_cols = numeric_cols
        self.means_ = None
        self.stds_ = None

    def fit(self, X, y=None):
        if self.numeric_cols is None:
            exclude_cols = {self.id_col, 'sleep_date', 'lifelog_date', 'date', 'timestamp'}
            self.numeric_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

        # Fit statistics strictly on X (assumed to be Train fold only)
        self.means_ = X.groupby(self.id_col)[self.numeric_cols].mean()
        self.stds_ = X.groupby(self.id_col)[self.numeric_cols].std().replace(0.0, np.nan).fillna(1.0)
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Vectorized mapping to prevent Pandas fragmentation warnings
        mapped_means = self.means_.reindex(X_copy[self.id_col]).fillna(0.0).values
        mapped_stds = self.stds_.reindex(X_copy[self.id_col]).fillna(1.0).values

        # Calculate subject-specific Z-Score
        zscore_vals = (X_copy[self.numeric_cols].values - mapped_means) / (mapped_stds + 1e-6)

        # Concatenate Z-Score features
        zscore_df = pd.DataFrame(
            zscore_vals,
            columns=[f"{col}_subject_zscore" for col in self.numeric_cols],
            index=X_copy.index
        )
        return pd.concat([X_copy, zscore_df], axis=1)

class FoldWisePCA(BaseEstimator, TransformerMixin):
    """
    A strictly leak-free PCA dimensionality reduction transformer
    that fits a SimpleImputer and PCA model ONLY on the train folds.
    """
    def __init__(self, n_components=5, random_state=42, numeric_cols=None):
        self.n_components = n_components
        self.random_state = random_state
        self.numeric_cols = numeric_cols
        self.imputer = None
        self.pca = None

    def fit(self, X, y=None):
        if self.numeric_cols is None:
            exclude_cols = {'subject_id', 'sleep_date', 'lifelog_date', 'date', 'timestamp'}
            self.numeric_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

        # Keep columns that have at least some non-null values in fitting
        self.numeric_cols = [c for c in self.numeric_cols if X[c].notna().sum() > 0]

        if len(self.numeric_cols) >= self.n_components:
            self.imputer = SimpleImputer(strategy='mean')
            self.pca = PCA(n_components=self.n_components, random_state=self.random_state)

            imputed_vals = self.imputer.fit_transform(X[self.numeric_cols])
            self.pca.fit(imputed_vals)
        return self

    def transform(self, X):
        X_copy = X.copy()
        if self.pca is not None and self.imputer is not None:
            imputed_vals = self.imputer.transform(X_copy[self.numeric_cols])
            pca_vals = self.pca.transform(imputed_vals)

            for idx in range(self.n_components):
                X_copy[f"pca_comp_{idx}"] = pca_vals[:, idx]
        return X_copy

class LeakFreeSubjectTargetEncoder(BaseEstimator, TransformerMixin):
    """
    A strictly leak-free target encoder for Subject ID (or user_id)
    utilizing Laplace smoothing fit ONLY on the train target variables.
    """
    def __init__(self, id_col='subject_id', m_smooth=10.0):
        self.id_col = id_col
        self.m_smooth = m_smooth
        self.global_mean_ = 0.5
        self.subject_te_ = None

    def fit(self, X, y):
        # Fit strictly on train target label y
        y_series = pd.Series(y, index=X.index)
        self.global_mean_ = y_series.fillna(0.5).mean()

        non_null_mask = y_series.notna()
        group_sums = y_series[non_null_mask].groupby(X.loc[non_null_mask, self.id_col]).sum()
        group_counts = y_series[non_null_mask].groupby(X.loc[non_null_mask, self.id_col]).count()

        self.subject_te_ = (group_sums + self.m_smooth * self.global_mean_) / (group_counts + self.m_smooth)
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy["subject_target_encoded"] = X_copy[self.id_col].map(self.subject_te_).fillna(self.global_mean_)
        return X_copy


# ==============================================================================
# [10-Fold Subject Hole 교차 검증기 및 무결성 검증 엔진 전체 소스 코드]
# ==============================================================================
import logging

logger = logging.getLogger(__name__)

def clip_probabilities(df: pd.DataFrame, target_cols: list, eps: float = 1e-15) -> pd.DataFrame:
    """
    Log-Loss is extremely sensitive to confident but wrong predictions.
    Clipping probabilities away from absolute 0 and 1 prevents infinite loss.
    """
    df_copy = df.copy()
    for col in target_cols:
        if col in df_copy.columns:
            df_copy[col] = np.clip(df_copy[col], eps, 1 - eps)
    return df_copy

def validate_submission(sub_df: pd.DataFrame, sample_sub_df: pd.DataFrame) -> bool:
    """
    Strictly validates the generated submission against the required sample submission format.
    Checks for:
    1. Same number of rows
    2. Same columns
    3. Same IDs (if applicable)
    4. Values are within [0, 1] range
    """
    is_valid = True

    # 1. Check Row Count
    if len(sub_df) != len(sample_sub_df):
        logger.error(f"Validation Failed: Row count mismatch. Expected {len(sample_sub_df)}, got {len(sub_df)}.")
        is_valid = False

    # 2. Check Columns
    expected_cols = set(sample_sub_df.columns)
    actual_cols = set(sub_df.columns)
    if expected_cols != actual_cols:
        logger.error(f"Validation Failed: Column mismatch. Expected {expected_cols}, got {actual_cols}.")
        is_valid = False

    # 3. Check value ranges for probability columns (assuming all non-ID columns are probabilities)
    # Exclude typical ID columns
    id_cols = ['subject_id', 'user_id', 'date', 'timestamp']
    prob_cols = [c for c in sub_df.columns if c not in id_cols]

    for col in prob_cols:
        if sub_df[col].dtype.kind in 'fc': # Float or complex
            min_val = sub_df[col].min()
            max_val = sub_df[col].max()
            if min_val < 0.0 or max_val > 1.0:
                logger.error(f"Validation Failed: Values in column {col} are out of bounds [0, 1]. Min: {min_val}, Max: {max_val}.")
                is_valid = False

    if is_valid:
        logger.info("Submission passed all validation checks!")

    return is_valid

def make_subject_timeseries_folds(train: pd.DataFrame, n_folds: int = 5) -> list:
    """
    Subject 내에서 엄격한 TimeSeries 분할을 수행하는 CV 전략
    미래 데이터를 참조하는 누수(Look-ahead bias)를 차단하면서,
    충분한 학습 데이터와 검증 데이터를 확보하기 위해 TimeSeriesSplit을 사용합니다.
    """
    from sklearn.model_selection import TimeSeriesSplit
    result = [([], []) for _ in range(n_folds)]

    sort_col = "sleep_date" if "sleep_date" in train.columns else "date"
    id_col = "subject_id" if "subject_id" in train.columns else "user_id"

    for subject_id, group in train.sort_values([id_col, sort_col]).groupby(id_col, sort=False):
        indices = group.index.to_numpy()
        n_samples = len(indices)

        # If a user has too few records, just put them all in train for later folds
        if n_samples < n_folds + 1:
            for fold_idx in range(n_folds):
                result[fold_idx][0].extend(indices)
            continue

        # We want to ensure each fold has a reasonable test size without starving the train set
        test_size = max(1, n_samples // (n_folds + 2))
        tscv = TimeSeriesSplit(n_splits=n_folds, test_size=test_size)

        for fold_idx, (t_idx, v_idx) in enumerate(tscv.split(indices)):
            result[fold_idx][0].extend(indices[t_idx])
            result[fold_idx][1].extend(indices[v_idx])

    # Convert lists to numpy arrays
    final_result = []
    for t_idx, v_idx in result:
        if len(t_idx) > 0 and len(v_idx) > 0:
            final_result.append((np.array(t_idx), np.array(v_idx)))

    return final_result

def make_subject_hole_folds(train: pd.DataFrame, n_folds: int = 5) -> list:
    """
    Subject 내에서 시간적 홀(hole)을 만들어 interleaved Train/Test 구조를 시뮬레이션하는 CV 전략.
    
    Args:
        train: 훈련 데이터프레임
        n_folds: CV 폴드 수
    
    Returns:
        (train_idx, valid_idx) numpy array 튜플 리스트
    """
    all_indices = train.index.to_numpy()
    result = []

    # 각 Subject별 시간순 정렬 후 block_count 개수로 분할
    block_count = max(n_folds * 2, 4)  # 홀(hole)을 생성하기 위해 fold 수의 2배로 설정
    by_subject = {}

    sort_col = "sleep_date" if "sleep_date" in train.columns else "date"
    id_col = "subject_id" if "subject_id" in train.columns else "user_id"

    for subject_id, group in train.sort_values([id_col, sort_col]).groupby(id_col, sort=False):
        indices = group.index.to_numpy()
        # 동일 크기로 분할
        chunks = [chunk for chunk in np.array_split(indices, block_count) if len(chunk)]
        by_subject[str(subject_id)] = chunks

    # 각 fold에 대해 두 개의 홀 생성 (early hole, late hole)
    for fold_id in range(n_folds):
        valid_parts = []

        for chunks in by_subject.values():
            # fold_id: early hole, fold_id + n_folds: late hole
            for hole_id in (fold_id, fold_id + n_folds):
                if hole_id < len(chunks):
                    valid_parts.append(chunks[hole_id])

        if not valid_parts:
            continue

        valid_idx = np.concatenate(valid_parts)
        train_idx = np.setdiff1d(all_indices, valid_idx, assume_unique=False)

        if len(train_idx) > 0 and len(valid_idx) > 0:
            result.append((train_idx, valid_idx))

    return result




## 🧬 Part 3: 고성능 멀티모달 피처 엔지니어링 핵심 모듈 전체 소스 코드
*   스마트폰과 스마트워치 센서로부터 심박수(HR) EWM(지수이동평균) 정규화 변동성, 밤낮 조도 대비 비율, 앱 사용 엔트로피를 가공하는 모든 핵심 연산 소스 코드입니다.
*   유저별 missing calendar day를 resample('D')로 온전히 복구하여 시계열 데이터의 왜곡을 방어합니다.
*   **노트북의 완전한 투명성을 위해 피처 엔지니어링 전체 소스코드를 셀 안에 완전히 임베딩하였습니다.**

In [ ]:
# ==============================================================================
# [피처 엔지니어링 파이프라인 전체 소스 코드]
# ==============================================================================
import pandas as pd


def aggregate_time_series(df: pd.DataFrame, time_col: str, agg_col: str, freq: str = "D") -> pd.DataFrame:
    """
    Aggregates time-series data at a given frequency.
    """
    df_copy = df.copy()
    if time_col in df_copy.columns:
        df_copy[time_col] = pd.to_datetime(df_copy[time_col])
        df_copy = df_copy.set_index(time_col)
        # Assuming numeric columns for simple aggregation
        numeric_cols = df_copy.select_dtypes(include=[np.number]).columns
        df_agg = df_copy[numeric_cols].resample(freq).mean().reset_index()
        return df_agg
    return df_copy


def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Handles missing values via leak-free forward filling
    and adding missingness indicator features.
    """
    df_copy = df.copy()
    # Create missingness indicators
    for col in df_copy.columns:
        if df_copy[col].isnull().any():
            df_copy[f"{col}_is_missing"] = df_copy[col].isnull().astype(int)

    # STRICT LEAKAGE PREVENTION: Only apply forward-fill per subject chronologically.
    # Linear interpolation is banned as it leaks future values into the past.
    numeric_cols = df_copy.select_dtypes(include=[np.number]).columns
    id_col = "subject_id" if "subject_id" in df_copy.columns else "user_id"

    # SOTA-v13: 3일 이상 장기 센서 유실 시 강제 전파를 막기 위해 ffill(limit=2) 로 보간 범위 엄격 제한!
    # 무작정 오래된 데이터를 복제하여 주입하는 Preprocessing Poisoning을 원천 방지하고,
    # 결측 정보는 결측 상태(NaN) 그대로 보존하여 LightGBM/CatBoost의 Default Routing에 맡깁니다.
    if id_col in df_copy.columns:
        df_copy[numeric_cols] = df_copy.groupby(id_col)[numeric_cols].transform(
            lambda x: x.ffill(limit=2)
        )
    else:
        df_copy[numeric_cols] = df_copy[numeric_cols].ffill(limit=2)
    return df_copy


def add_cross_sensor_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates interaction features between different sensors.
    """
    df_copy = df.copy()

    # 1. Heart Rate vs Activity (Arousal vs Exertion)
    if 'ch2025_wHr_hr_mean_mean' in df_copy.columns and 'ch2025_mActivity_m_activity_mean' in df_copy.columns:
        # High HR but low activity might indicate stress/arousal without physical exertion
        df_copy['stress_proxy_hr_act'] = df_copy['ch2025_wHr_hr_mean_mean'] / (df_copy['ch2025_mActivity_m_activity_mean'] + 1.0)

    # 2. Light vs Screen (Blue light exposure)
    light_col = next((c for c in df_copy.columns if 'light_mean' in c), None)
    if light_col and 'ch2025_mScreenStatus_m_screen_use_mean' in df_copy.columns:
        # Screen use when ambient light is low (nighttime screen use)
        df_copy['night_screen_proxy'] = df_copy['ch2025_mScreenStatus_m_screen_use_mean'] / (df_copy[light_col] + 1.0)

    # 3. Circadian Rhythm: HR Dip Ratio (Night / Day)
    hr_night_col = next((c for c in df_copy.columns if 'hr_night_mean' in c), None)
    hr_day_col = next((c for c in df_copy.columns if 'hr_day_mean' in c), None)
    if hr_night_col and hr_day_col:
        df_copy['hr_dip_ratio'] = df_copy[hr_night_col] / (df_copy[hr_day_col] + 1.0)

    # 4. Routine Stability (Wifi BSSID Jaccard Similarity)
    id_col = "subject_id" if "subject_id" in df_copy.columns else "user_id"
    bssid_col = next((c for c in df_copy.columns if 'bssid_set' in c), None)
    if bssid_col:
        # Calculate jaccard similarity with previous day for each user
        df_copy['bssid_set_lag1'] = df_copy.groupby(id_col)[bssid_col].shift(1)

        def calc_jaccard(row):
            set1 = row[bssid_col]
            set2 = row['bssid_set_lag1']
            if isinstance(set1, set) and isinstance(set2, set) and (len(set1) > 0 or len(set2) > 0):
                intersection = len(set1.intersection(set2))
                union = len(set1.union(set2))
                return intersection / union
            return 0.0

        df_copy['routine_stability_jaccard'] = df_copy.apply(calc_jaccard, axis=1)
        df_copy = df_copy.drop(columns=[bssid_col, 'bssid_set_lag1'], errors='ignore')

    # 5. Physical Exhaustion Index (GPS Distance * Steps)
    gps_dist_col = next((c for c in df_copy.columns if 'gps_dist' in c), None)
    steps_col = next((c for c in df_copy.columns if 'step_sum' in c or 'ch2025_wPedo_step_sum' in c), None)
    if gps_dist_col and steps_col:
        df_copy['physical_exhaustion'] = df_copy[gps_dist_col] * df_copy[steps_col]

    # 6. Deep Interactions
    hr_col = next((c for c in df_copy.columns if 'hr_mean' in c and not c.endswith('_zscore')), None)
    act_col = next((c for c in df_copy.columns if 'm_activity_mean' in c and not c.endswith('_zscore')), None)
    if hr_col and act_col:
        # High HR but low activity = Stress/Anxiety proxy
        df_copy['stress_proxy_hr_act'] = df_copy[hr_col] / (df_copy[act_col] + 1e-5)

    # 7. Weekend Transitions
    if "dayofweek" in df_copy.columns:
        df_copy['is_monday'] = (df_copy['dayofweek'] == 0).astype(int)
        df_copy['is_friday'] = (df_copy['dayofweek'] == 4).astype(int)

    return df_copy


def apply_personalized_normalization(df: pd.DataFrame, user_col: str, feature_cols: list) -> pd.DataFrame:
    """
    Normalizes features per user (Z-score normalization).
    Uses expanding window to prevent data leakage from future days.
    Preserves raw features for objective targets (S1-S4) while providing
    Z-scores for subjective targets (Q1-Q3).
    """
    df_copy = df.copy()
    if user_col in df_copy.columns:
        # DO NOT normalize target columns
        exclude_cols = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4", "target", "date", "timestamp"]
        existing_cols = [c for c in feature_cols if c in df_copy.columns and c not in exclude_cols]

        if existing_cols:
            grouped = df_copy.groupby(user_col)[existing_cols]
            # EWM ensures day N is normalized giving more weight to recent days (span=14)
            means = grouped.ewm(span=14, min_periods=1).mean().reset_index(level=0, drop=True)
            stds = grouped.ewm(span=14, min_periods=1).std().reset_index(level=0, drop=True)

            # For the first day, stds will be NaN. We must fill it with 1.0 (or 0) so z-score doesn't become NaN
            stds = stds.fillna(1.0)

            # --- Z-Score Warm-up Correction ---
            # Compute cumulative count per user to track their time scale (starts at 1)
            cum_counts = df_copy.groupby(user_col).cumcount() + 1
            # Add a regularization buffer for early days (cum_counts < 5) to prevent extreme outlier scales
            # The buffer smoothly ramps down from 0.8 (on day 1) to 0.0 (on day 5+)
            reg_buffer = (5.0 - cum_counts).clip(lower=0.0) * 0.2
            # Z-scores computation with soft-shrinkage safety scale
            zscores = (df_copy[existing_cols] - means) / (stds + reg_buffer.values[:, None] + 1e-8)
            # -----------------------------------

            zscores.columns = [f"{col}_zscore" for col in existing_cols]

            # Concat z-scores alongside raw features
            df_copy = pd.concat([df_copy, zscores], axis=1)
    return df_copy


def add_rolling_features(df: pd.DataFrame, user_col: str, date_col: str, windows: list = [3, 7, 14, 21]) -> pd.DataFrame:
    """
    Adds time-delta-correct rolling mean, std, lag, diff and genuine EMA features per user.
    Uses resample('D') to reconstruct explicit chronological calendars, preventing 
    temporal distortion due to missing days of watch tracking.
    """
    df_copy = df.copy()
    if user_col not in df_copy.columns or date_col not in df_copy.columns:
        return df_copy

    df_copy[date_col] = pd.to_datetime(df_copy[date_col])
    df_copy = df_copy.sort_values(by=[user_col, date_col])

    # Select numerical columns that are not already rolling features or targets/IDs
    exclude_cols = [user_col, date_col, "target", "Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]
    numeric_cols = [c for c in df_copy.select_dtypes(include=[np.number]).columns if c not in exclude_cols and not c.endswith("_is_missing")]

    roll_features_list = []

    # Apply user-wise Resampling to safely resolve Time-Delta gap anomalies
    for user, group in df_copy.groupby(user_col):
        # Sort and resample daily to maintain a linear, continuous time scale
        group_ts = group.set_index(date_col).sort_index()
        # Aggregating daily duplicate readings safely by mean
        group_ts_daily = group_ts[numeric_cols].resample('D').mean()

        # Lag 1: Pure previous-day value with zero calendar look-ahead leak
        lag1 = group_ts_daily.shift(1)
        lag1.columns = [f"{c}_lag1" for c in numeric_cols]

        # Diff 1: Pure previous-day variation diff
        diff1 = group_ts_daily.diff(1)
        diff1.columns = [f"{c}_diff1" for c in numeric_cols]

        user_roll_dfs = [lag1, diff1]

        # Calendar-correct Rolling Statistics and true Exponential Moving Averages (EMA)
        for w in windows:
            roll = group_ts_daily.rolling(window=w, min_periods=1)

            mean_df = roll.mean()
            mean_df.columns = [f"{c}_roll{w}_mean" for c in numeric_cols]

            std_df = roll.std().fillna(0.0)
            std_df.columns = [f"{c}_roll{w}_std" for c in numeric_cols]

            # SOTA-v14: True EMA implementation to properly represent slowly deteriorating temporal states
            ema_df = group_ts_daily.ewm(span=w, min_periods=1).mean()
            ema_df.columns = [f"{c}_ema{w}_mean" for c in numeric_cols]

            user_roll_dfs.extend([mean_df, std_df, ema_df])

        merged_user_roll = pd.concat(user_roll_dfs, axis=1)
        merged_user_roll[user_col] = user
        merged_user_roll = merged_user_roll.reset_index()
        roll_features_list.append(merged_user_roll)

    all_roll_features = pd.concat(roll_features_list, ignore_index=True)
    all_roll_features[date_col] = pd.to_datetime(all_roll_features[date_col]).dt.date

    # Merge engineered rolling matrices back using absolute date mapping to prevent spatial shift leakage
    df_copy['date_only'] = pd.to_datetime(df_copy[date_col]).dt.date
    df_copy = pd.merge(
        df_copy,
        all_roll_features,
        left_on=[user_col, 'date_only'],
        right_on=[user_col, date_col],
        how='left',
        suffixes=('', '_roll_temp')
    )
    df_copy = df_copy.drop(columns=['date_only', date_col + '_roll_temp'], errors='ignore')

    return df_copy


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies comprehensive feature engineering to the given DataFrame
    based on the master plan strategies.
    """
    df_copy = df.copy()
    id_col = "subject_id" if "subject_id" in df_copy.columns else "user_id"
    date_col = "date"

    # STRICT TEMPORAL SORTING: Force sorting chronologically by user and date
    # before engineering features, so expanding window/ffill/rolling work chronological.
    if id_col in df_copy.columns and date_col in df_copy.columns:
        df_copy[date_col] = pd.to_datetime(df_copy[date_col])
        df_copy = df_copy.sort_values(by=[id_col, date_col]).reset_index(drop=True)
        # Biting leakage prevention: Assert chronological monotonicity per user to block time-reversed leakage
        assert df_copy.groupby(id_col)[date_col].apply(lambda x: x.is_monotonic_increasing).all(), \
            "CRITICAL: Chronological order sorting failed! Potential look-ahead leakage danger."

    if "date" in df_copy.columns:
        dt = pd.to_datetime(df_copy["date"])
        if "dayofweek" not in df_copy.columns:
            df_copy["dayofweek"] = dt.dt.dayofweek
        if "is_weekend" not in df_copy.columns:
            df_copy["is_weekend"] = df_copy["dayofweek"].isin([5, 6]).astype(int)

        df_copy["month"] = dt.dt.month
        df_copy["day"] = dt.dt.day

        # Cyclical encoding
        df_copy['dayofweek_sin'] = np.sin(2 * np.pi * df_copy['dayofweek'] / 7)
        df_copy['dayofweek_cos'] = np.cos(2 * np.pi * df_copy['dayofweek'] / 7)
        df_copy['month_sin'] = np.sin(2 * np.pi * df_copy['month'] / 12)
        df_copy['month_cos'] = np.cos(2 * np.pi * df_copy['month'] / 12)

    df_copy = handle_missing_values(df_copy)

    # Add Cross-Sensor interactions
    df_copy = add_cross_sensor_features(df_copy)

    id_col = "subject_id" if "subject_id" in df_copy.columns else "user_id"
    date_col = "date"

    # Add rolling and lag features before user normalization
    if id_col in df_copy.columns and date_col in df_copy.columns:
        df_copy = add_rolling_features(df_copy, user_col=id_col, date_col=date_col)

    # Placeholder for numerical feature columns to normalize per user
    feature_cols = [c for c in df_copy.select_dtypes(include=[np.number]).columns if not c.endswith("_is_missing") and c not in [id_col]]
    df_copy = apply_personalized_normalization(df_copy, user_col=id_col, feature_cols=feature_cols)

    return df_copy



## 🚀 Part 4: 6-Model 순차 분류 체인 파이프라인 학습 기동
XGBoost, LightGBM, CatBoost, ExtraTrees, RandomForest, LogisticRegression 등 6대 모델 블렌더를 sequential classifier chain 방식으로 학습시키고, 10-Fold Subject-Wise 교차 검증을 통해 최종 OOF(Out-of-Fold) 파일을 디스크에 생성하는 메인 스텝입니다.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

# Suppress warnings
warnings.filterwarnings('ignore')

# Add project root to sys.path
sys.path.append(str(Path(__file__).resolve().parents[1]))

from src.data.data_loader import DataLoader
from src.data.make_dataset import build_training_dataset
from src.features.build_features import engineer_features
from src.features.transformers import FoldWisePCA, LeakFreeSubjectTargetEncoder, LeakFreeSubjectZScaler
from src.utils.logger import get_logger
from src.utils.validation import make_subject_hole_folds

logger = get_logger(__name__)

def add_temporal_target_features(df, y, train_idx, target_col, id_col, half_life=3.0):
    df_copy = df.copy()
    df_copy['original_index'] = df_copy.index

    # Isolate training labels
    y_train = pd.Series(np.nan, index=df_copy.index)
    y_train.loc[train_idx] = y.loc[train_idx]
    df_copy['y_train_temp'] = y_train

    df_copy['date_temp'] = pd.to_datetime(df_copy['date'])
    df_copy = df_copy.sort_values(by=[id_col, 'date_temp']).reset_index(drop=True)

    # Shift to prevent target leakage
    df_copy['date_with_target'] = df_copy['date_temp'].where(df_copy['y_train_temp'].notna())
    df_copy['y_train_prev_source'] = df_copy.groupby(id_col)['y_train_temp'].shift(1)
    df_copy['date_prev_source'] = df_copy.groupby(id_col)['date_with_target'].shift(1)

    df_copy['y_train_next_source'] = df_copy.groupby(id_col)['y_train_temp'].shift(-1)
    df_copy['date_next_source'] = df_copy.groupby(id_col)['date_with_target'].shift(-1)

    df_copy['y_prev'] = df_copy.groupby(id_col)['y_train_prev_source'].ffill()
    df_copy['date_prev'] = df_copy.groupby(id_col)['date_prev_source'].ffill()
    df_copy['y_next'] = df_copy.groupby(id_col)['y_train_next_source'].bfill()
    df_copy['date_next'] = df_copy.groupby(id_col)['date_next_source'].bfill()

    df_copy['delta_prev'] = (df_copy['date_temp'] - pd.to_datetime(df_copy['date_prev'])).dt.days
    df_copy['delta_next'] = (pd.to_datetime(df_copy['date_next']) - df_copy['date_temp']).dt.days

    # Calculate subject-specific mean baseline (Laplace smoothed)
    subject_means = df_copy.groupby(id_col)['y_train_temp'].transform('mean').fillna(0.5)

    # Apply Exponential Decay Chronological Weighted Interpolation
    decay_lambda = np.log(2.0) / half_life
    w_prev = np.exp(-decay_lambda * df_copy['delta_prev'].fillna(999.0))
    w_next = np.exp(-decay_lambda * df_copy['delta_next'].fillna(999.0))

    prior_weight = 0.2
    denom = w_prev + w_next + prior_weight
    df_copy['y_interp'] = (w_prev * df_copy['y_prev'].fillna(0.5) +
                           w_next * df_copy['y_next'].fillna(0.5) +
                           prior_weight * subject_means) / denom

    df_copy['y_prev'] = df_copy['y_prev'].fillna(subject_means)
    df_copy['y_next'] = df_copy['y_next'].fillna(subject_means)
    df_copy['y_interp'] = df_copy['y_interp'].fillna(subject_means)

    df_copy = df_copy.sort_values(by='original_index').reset_index(drop=True)

    features = pd.DataFrame(index=df.index)
    features[f'{target_col}_prev'] = df_copy['y_prev'].values
    features[f'{target_col}_next'] = df_copy['y_next'].values
    features[f'{target_col}_interp'] = df_copy['y_interp'].values
    features[f'{target_col}_log_delta_prev'] = np.log1p(df_copy['delta_prev'].fillna(999.0)).values
    features[f'{target_col}_log_delta_next'] = np.log1p(df_copy['delta_next'].fillna(999.0)).values

    return features

def apply_robust_directives(df_train_fold, df_test_fold, tr_idx, target, id_col, numeric_cols, y_train_fold):
    """
    Applies 100% leak-free robust directives:
    1. For S2, S3, S4: Generate pairwise interactions (plus, minus, mul) for top 5 correlated candidate features.
       Skip division to prevent extreme value inflation.
    """
    df_train_fold = df_train_fold.copy()
    df_test_fold = df_test_fold.copy()

    special_cols = []

    if target in ["S2", "S3", "S4"]:
        # Find candidate columns to compute correlation
        candidate_cols = [c for c in df_train_fold.columns if c.endswith("_subject_zscore")]
        corrs = {}
        for col in candidate_cols:
            c = df_train_fold.iloc[tr_idx][col].corr(y_train_fold.iloc[tr_idx])
            if pd.notna(c):
                corrs[col] = abs(c)
        top_5 = sorted(corrs.items(), key=lambda x: x[1], reverse=True)[:5]
        top_5_cols = [col for col, _ in top_5]

        for i in range(len(top_5_cols)):
            for j in range(i + 1, len(top_5_cols)):
                c0, c1 = top_5_cols[i], top_5_cols[j]

                # Add
                name_add = f"{c0}_plus_{c1}"
                df_train_fold[name_add] = df_train_fold[c0] + df_train_fold[c1]
                df_test_fold[name_add] = df_test_fold[c0] + df_test_fold[c1]
                special_cols.append(name_add)

                # Subtract
                name_sub = f"{c0}_minus_{c1}"
                df_train_fold[name_sub] = df_train_fold[c0] - df_train_fold[c1]
                df_test_fold[name_sub] = df_test_fold[c0] - df_test_fold[c1]
                special_cols.append(name_sub)

                # Multiply
                name_mul = f"{c0}_mul_{c1}"
                df_train_fold[name_mul] = df_train_fold[c0] * df_train_fold[c1]
                df_test_fold[name_mul] = df_test_fold[c0] * df_test_fold[c1]
                special_cols.append(name_mul)

    return df_train_fold, df_test_fold, special_cols

def run_ultimate_pipeline():
    logger.info("Initializing Ultimate SOTA-v28 Sequential Classifier Chain Pipeline (6-Model Blender)...")

    # 1. Load Data
    data_loader = DataLoader(data_dir="data")
    metrics_df = data_loader.load_metrics()
    sample_sub_df = data_loader.load_sample_submission()

    processed_train_path = Path("data/processed/df_train_raw.parquet")
    processed_test_path = Path("data/processed/df_test_raw.parquet")

    if processed_train_path.exists() and processed_test_path.exists():
        logger.info("Found cached raw datasets in data/processed/. Loading directly for massive speedup...")
        df_train_raw = pd.read_parquet(processed_train_path)
        df_test_raw = pd.read_parquet(processed_test_path)
    else:
        logger.info("No cached datasets found. Rebuilding from raw sensor parquets...")
        sensor_data_dict = data_loader.load_sensor_data()
        df_train_raw = build_training_dataset(metrics_df, sensor_data_dict)
        df_test_raw = build_training_dataset(sample_sub_df, sensor_data_dict)

        # Save to cache
        processed_train_path.parent.mkdir(parents=True, exist_ok=True)
        df_train_raw.to_parquet(processed_train_path, index=False)
        df_test_raw.to_parquet(processed_test_path, index=False)
        logger.info("Cached raw datasets to data/processed/ for future SOTA runs.")

    df_train_raw['is_test'] = 0
    df_test_raw['is_test'] = 1
    df_all_raw = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

    # 2. Engineer Features
    df_all = engineer_features(df_all_raw)

    df_train = df_all[df_all['is_test'] == 0].drop(columns=['is_test']).copy()
    df_test = df_all[df_all['is_test'] == 1].drop(columns=['is_test']).copy()

    id_col = "subject_id" if "subject_id" in df_train.columns else "user_id"
    targets = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

    # 3. Anchor Stacking: Incorporate Deep Learning Predictions (LSTM/Attention)
    lstm_oof_path = 'submissions/oof_dl_v10.csv'
    lstm_sub_path = 'submissions/submission_dl_v10.csv'

    base_feature_cols = []
    if os.path.exists(lstm_oof_path) and os.path.exists(lstm_sub_path):
        logger.info("Deep Learning sequence models found. Merging as anchor stack features...")
        base_oof = pd.read_csv(lstm_oof_path)
        base_oof['date'] = pd.to_datetime(base_oof['date']).dt.date
        df_train['date'] = pd.to_datetime(df_train['date']).dt.date
        df_train = pd.merge(df_train, base_oof, on=[id_col, 'date'], how='left', suffixes=('', '_base'))

        base_sub = pd.read_csv(lstm_sub_path)
        if 'date' not in base_sub.columns:
            if 'lifelog_date' in base_sub.columns:
                base_sub['date'] = base_sub['lifelog_date']
            elif 'sleep_date' in base_sub.columns:
                base_sub['date'] = base_sub['sleep_date']

        base_sub['date'] = pd.to_datetime(base_sub['date']).dt.date
        df_test['date'] = pd.to_datetime(df_test['date']).dt.date
        df_test = pd.merge(df_test, base_sub, on=[id_col, 'date'], how='left', suffixes=('', '_base'))

        eps = 1e-6
        for t in targets:
            base_col = f"{t}_base"
            if base_col in df_train.columns:
                prob = df_train[base_col].fillna(0.5).clip(eps, 1-eps)
                df_train[f'base_{t}_prob'] = prob
                df_train[f'base_{t}_logit'] = np.log(prob / (1 - prob))
            if base_col in df_test.columns:
                prob_t = df_test[base_col].fillna(0.5).clip(eps, 1-eps)
                df_test[f'base_{t}_prob'] = prob_t
                df_test[f'base_{t}_logit'] = np.log(prob_t / (1 - prob_t))
            base_feature_cols.extend([f'base_{t}_prob', f'base_{t}_logit'])

    # Setup CV
    n_splits = 5
    # Sorting train dataset chronologically per subject is critical for temporal mapping
    df_train = df_train.sort_values(by=[id_col, 'date']).reset_index(drop=True)
    df_test = df_test.sort_values(by=[id_col, 'date']).reset_index(drop=True)

    cv_folds = make_subject_hole_folds(df_train, n_folds=n_splits)

    drop_cols = targets + [id_col, "date", "timestamp", "sleep_date", "lifelog_date"]
    for t in targets:
        drop_cols.extend([f"{t}_zscore", f"{t}_is_missing"])
        if f"{t}_base" in df_train.columns:
            drop_cols.append(f"{t}_base")

    feature_cols = [c for c in df_train.columns if c not in drop_cols and c not in base_feature_cols]
    numeric_cols = df_train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

    # 6 Prediction arrays
    oof_preds_cat = np.zeros((len(df_train), len(targets)))
    oof_preds_lgb = np.zeros((len(df_train), len(targets)))
    oof_preds_xgb = np.zeros((len(df_train), len(targets)))
    oof_preds_et = np.zeros((len(df_train), len(targets)))
    oof_preds_rf = np.zeros((len(df_train), len(targets)))
    oof_preds_lr = np.zeros((len(df_train), len(targets)))

    test_preds_cat = np.zeros((len(df_test), len(targets)))
    test_preds_lgb = np.zeros((len(df_test), len(targets)))
    test_preds_xgb = np.zeros((len(df_test), len(targets)))
    test_preds_et = np.zeros((len(df_test), len(targets)))
    test_preds_rf = np.zeros((len(df_test), len(targets)))
    test_preds_lr = np.zeros((len(df_test), len(targets)))

    # Define Easiest-to-Hardest Sequential Classifier Chain order
    chain_order = ["S3", "S2", "S1", "Q3", "Q2", "S4", "Q1"]

    # ==========================================
    # RUN SEQUENTIAL CLASSIFIER CHAIN
    # ==========================================
    logger.info("========== Starting 6-Model Sequential Classifier Chain Training ==========")
    for chain_step, target in enumerate(chain_order):
        t_global_idx = targets.index(target)
        logger.info(f"\n--- [Chain Step {chain_step+1}/7] Target: {target} ---")

        # 1. Inject previously predicted targets from the chain as features (fully out-of-fold!)
        prev_targets = chain_order[:chain_step]
        chain_feature_cols = []
        for prev_t in prev_targets:
            prev_global_idx = targets.index(prev_t)

            # 6-Model Blender OOF & Test
            blend_prev_oof = (1/6.0) * (oof_preds_cat[:, prev_global_idx] +
                                        oof_preds_lgb[:, prev_global_idx] +
                                        oof_preds_xgb[:, prev_global_idx] +
                                        oof_preds_et[:, prev_global_idx] +
                                        oof_preds_rf[:, prev_global_idx] +
                                        oof_preds_lr[:, prev_global_idx])

            blend_prev_test = (1/6.0) * (test_preds_cat[:, prev_global_idx] +
                                         test_preds_lgb[:, prev_global_idx] +
                                         test_preds_xgb[:, prev_global_idx] +
                                         test_preds_et[:, prev_global_idx] +
                                         test_preds_rf[:, prev_global_idx] +
                                         test_preds_lr[:, prev_global_idx])

            df_train[f'meta_prob_{prev_t}'] = blend_prev_oof
            df_test[f'meta_prob_{prev_t}'] = blend_prev_test

            eps = 1e-6
            df_train[f'meta_logit_{prev_t}'] = np.log(df_train[f'meta_prob_{prev_t}'].clip(eps, 1-eps) / (1 - df_train[f'meta_prob_{prev_t}'].clip(eps, 1-eps)))
            df_test[f'meta_logit_{prev_t}'] = np.log(df_test[f'meta_prob_{prev_t}'].clip(eps, 1-eps) / (1 - df_test[f'meta_prob_{prev_t}'].clip(eps, 1-eps)))

            chain_feature_cols.extend([f'meta_prob_{prev_t}', f'meta_logit_{prev_t}'])

        y = df_train[target]
        has_label = y.notna()

        valid_indices = np.where(has_label)[0]
        valid_idx_set = set(valid_indices)

        for fold, (train_idx, val_idx) in enumerate(cv_folds):
            tr_idx = np.array([i for i in train_idx if i in valid_idx_set])
            va_idx = np.array([i for i in val_idx if i in valid_idx_set])

            if len(tr_idx) == 0 or len(va_idx) == 0:
                continue

            df_train_fold = df_train.copy()
            y_train_fold = y.copy()

            # 0% Leak-Free Exponential Chronological Interpolation
            temp_features = add_temporal_target_features(df_train_fold, y_train_fold, tr_idx, target, id_col)

            temp_test = add_temporal_target_features(
                pd.concat([df_train_fold, df_test], ignore_index=True),
                pd.concat([y_train_fold, pd.Series(index=df_test.index)], ignore_index=True),
                np.arange(len(df_train_fold)),
                target,
                id_col
            )
            temp_features_test = temp_test.iloc[len(df_train_fold):].reset_index(drop=True)

            df_train_fold = pd.concat([df_train_fold, temp_features], axis=1)
            df_test_fold = pd.concat([df_test, temp_features_test], axis=1)

            # Subject scaling
            z_scaler = LeakFreeSubjectZScaler(id_col=id_col, numeric_cols=numeric_cols)
            z_scaler.fit(df_train_fold.iloc[tr_idx])
            df_train_fold = z_scaler.transform(df_train_fold)
            df_test_fold = z_scaler.transform(df_test_fold)

            # Laplace Target Encoding
            t_encoder = LeakFreeSubjectTargetEncoder(id_col=id_col)
            t_encoder.fit(df_train_fold.iloc[tr_idx], y_train_fold.iloc[tr_idx])
            df_train_fold = t_encoder.transform(df_train_fold)
            df_test_fold = t_encoder.transform(df_test_fold)

            # Apply Robust Directives (Pairwise Interactions for S2-S4)
            df_train_fold, df_test_fold, special_cols = apply_robust_directives(
                df_train_fold, df_test_fold, tr_idx, target, id_col, numeric_cols, y_train_fold
            )

            # PCA (for S2, S3, S4)
            pca_cols = []
            if target in ["S2", "S3", "S4"]:
                pca_transformer = FoldWisePCA(n_components=5, numeric_cols=numeric_cols)
                pca_transformer.fit(df_train_fold.iloc[tr_idx])
                df_train_fold = pca_transformer.transform(df_train_fold)
                df_test_fold = pca_transformer.transform(df_test_fold)
                pca_cols = [f"pca_comp_{idx}" for idx in range(5)]

            # Feature Correlation Selection
            stability_scores = {}
            if target in ["S1", "S2", "S3", "S4"]:
                candidate_cols = [c for c in df_train_fold.columns if c not in drop_cols and c not in base_feature_cols and not c.endswith("_freq") and "_plus_" not in c and "_minus_" not in c and "_mul_" not in c] + special_cols
            else:
                candidate_cols = [c for c in df_train_fold.columns if c.endswith("_subject_zscore")] + special_cols

            for feat in candidate_cols:
                corr = df_train_fold.iloc[tr_idx][feat].corr(y_train_fold.iloc[tr_idx])
                if pd.notna(corr):
                    stability_scores[feat] = abs(corr)

            selected_candidates = [k for k, v in stability_scores.items() if v >= 0.15]
            if len(selected_candidates) < 30:
                sorted_feats = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)
                selected_candidates = [f for f, v in sorted_feats[:80] if v >= 0.05]
            else:
                # Cap maximum candidates to prevent GBDT overfitting
                sorted_feats = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)
                selected_candidates = [f for f, v in sorted_feats[:80] if v >= 0.10]

            temporal_cols = [f'{target}_prev', f'{target}_next', f'{target}_interp', f'{target}_log_delta_prev', f'{target}_log_delta_next']
            active_cols = selected_candidates + base_feature_cols + chain_feature_cols + ["subject_target_encoded"] + pca_cols + temporal_cols + special_cols

            # Remove duplicates preserving order
            seen = set()
            active_cols = [x for x in active_cols if not (x in seen or seen.add(x))]

            X_tr = df_train_fold.iloc[tr_idx][active_cols].fillna(0.5)
            y_tr = y_train_fold.iloc[tr_idx]
            X_va = df_train_fold.iloc[va_idx][active_cols].fillna(0.5)
            X_te = df_test_fold[active_cols].fillna(0.5)

            # 1. CatBoost (Highly regularized, low depth)
            cat_model = CatBoostClassifier(iterations=250, depth=3, learning_rate=0.02, l2_leaf_reg=20.0, random_state=42, verbose=0, thread_count=2)
            cat_model.fit(X_tr, y_tr)
            oof_preds_cat[va_idx, t_global_idx] = cat_model.predict_proba(X_va)[:, 1]
            test_preds_cat[:, t_global_idx] += cat_model.predict_proba(X_te)[:, 1] / n_splits

            # 2. LightGBM (Highly regularized, low depth)
            lgb_model = lgb.LGBMClassifier(n_estimators=150, max_depth=3, learning_rate=0.02, min_child_samples=10, reg_alpha=5.0, reg_lambda=5.0, random_state=42, verbose=-1, n_jobs=2)
            lgb_model.fit(X_tr, y_tr)
            oof_preds_lgb[va_idx, t_global_idx] = lgb_model.predict_proba(X_va)[:, 1]
            test_preds_lgb[:, t_global_idx] += lgb_model.predict_proba(X_te)[:, 1] / n_splits

            # 3. XGBoost (Highly regularized, low depth)
            xgb_model = xgb.XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.02, reg_alpha=5.0, reg_lambda=5.0, random_state=42, eval_metric="logloss", n_jobs=2)
            xgb_model.fit(X_tr, y_tr)
            oof_preds_xgb[va_idx, t_global_idx] = xgb_model.predict_proba(X_va)[:, 1]
            test_preds_xgb[:, t_global_idx] += xgb_model.predict_proba(X_te)[:, 1] / n_splits

            # 4. ExtraTrees (More robust bagging parameters)
            et_model = ExtraTreesClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=2)
            et_model.fit(X_tr, y_tr)
            oof_preds_et[va_idx, t_global_idx] = et_model.predict_proba(X_va)[:, 1]
            test_preds_et[:, t_global_idx] += et_model.predict_proba(X_te)[:, 1] / n_splits

            # 5. RandomForest (More robust bagging parameters)
            rf_model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=2)
            rf_model.fit(X_tr, y_tr)
            oof_preds_rf[va_idx, t_global_idx] = rf_model.predict_proba(X_va)[:, 1]
            test_preds_rf[:, t_global_idx] += rf_model.predict_proba(X_te)[:, 1] / n_splits

            # 6. LogisticRegression (Robust Linear Model)
            lr_model = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
            lr_model.fit(X_tr, y_tr)
            oof_preds_lr[va_idx, t_global_idx] = lr_model.predict_proba(X_va)[:, 1]
            test_preds_lr[:, t_global_idx] += lr_model.predict_proba(X_te)[:, 1] / n_splits

        logger.info(f"Target {target} CV -> CatBoost: {log_loss(y[has_label], oof_preds_cat[has_label, t_global_idx]):.4f} | LightGBM: {log_loss(y[has_label], oof_preds_lgb[has_label, t_global_idx]):.4f} | LogisticRegression: {log_loss(y[has_label], oof_preds_lr[has_label, t_global_idx]):.4f} | ExtraTrees: {log_loss(y[has_label], oof_preds_et[has_label, t_global_idx]):.4f}")

    # Save OOFs and Submissions
    logger.info("Saving corrected OOF predictions...")

    oof_template = pd.DataFrame({'date': df_train['date'], 'subject_id': df_train[id_col]})
    sub_template = pd.DataFrame({'date': df_test['date'], 'subject_id': df_test[id_col]})

    cat_oof = oof_template.copy()
    cat_sub = sub_template.copy()
    cat_oof[targets] = oof_preds_cat
    cat_sub[targets] = test_preds_cat
    cat_oof.to_csv("submissions/oof_catboost_v10.csv", index=False)
    cat_sub.to_csv("submissions/submission_catboost_v10.csv", index=False)

    lgb_oof = oof_template.copy()
    lgb_sub = sub_template.copy()
    lgb_oof[targets] = oof_preds_lgb
    lgb_sub[targets] = test_preds_lgb
    lgb_oof.to_csv("submissions/oof_lgbm_v10.csv", index=False)
    lgb_sub.to_csv("submissions/submission_lgbm_v10.csv", index=False)

    xgb_oof = oof_template.copy()
    xgb_sub = sub_template.copy()
    xgb_oof[targets] = oof_preds_xgb
    xgb_sub[targets] = test_preds_xgb
    xgb_oof.to_csv("submissions/oof_xgb_v10.csv", index=False)
    xgb_sub.to_csv("submissions/submission_xgb_v10.csv", index=False)

    et_oof = oof_template.copy()
    et_sub = sub_template.copy()
    et_oof[targets] = oof_preds_et
    et_sub[targets] = test_preds_et
    et_oof.to_csv("submissions/oof_et_v10.csv", index=False)
    et_sub.to_csv("submissions/submission_et_v10.csv", index=False)

    rf_oof = oof_template.copy()
    rf_sub = sub_template.copy()
    rf_oof[targets] = oof_preds_rf
    rf_sub[targets] = test_preds_rf
    rf_oof.to_csv("submissions/oof_rf_v10.csv", index=False)
    rf_sub.to_csv("submissions/submission_rf_v10.csv", index=False)

    lr_oof = oof_template.copy()
    lr_sub = sub_template.copy()
    lr_oof[targets] = oof_preds_lr
    lr_sub[targets] = test_preds_lr
    lr_oof.to_csv("submissions/oof_lr_v10.csv", index=False)
    lr_sub.to_csv("submissions/submission_lr_v10.csv", index=False)

    logger.info("Ultimate SOTA-v28 temporal sequential neighborhood chain model successfully trained!")

if __name__ == "__main__":
    run_ultimate_pipeline()


## ⚖️ Part 5: 하이브리드 메타 스태킹 & 유저별 역사적 캘리브레이션
완성된 개별 모델들의 OOF 파일을 입력으로 받아,
1. 아웃오브폴드(Out-of-fold) 방식으로 **주체별 베타 캘리브레이션(Subject-wise Beta Calibration)**을 수행하여 확률 로짓 공간을 교정합니다.
2. 수치해석적 **SLSQP 가중 블렌딩**과 **L2 Regularized Logistic Stacking**을 교차 검증으로 비교하여 더 우수한 조합을 자동 낙점합니다.
3. 정서적 주관 지표(Q1~Q3) 성능을 극대화하기 위해, 유저별 역사적 사전 확률(Subject smoothed mean prior)과의 **최적 선형 검색 볼록 결합(Alpha Blending)**을 최종 릴리즈합니다.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

# Suppress warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path.cwd()))
from src.utils.logger import get_logger

logger = get_logger(__name__)

def calibrate_beta_subject_wise(y_true, p_raw, subjects, global_T=1.0):
    eps = 1e-15
    p_raw = np.clip(p_raw, eps, 1 - eps)
    logits = np.log(p_raw / (1 - p_raw))

    unique_subjects = np.unique(subjects)
    calibrated_p = np.zeros_like(p_raw)

    for subject in unique_subjects:
        subj_mask = subjects == subject
        other_mask = ~subj_mask

        if other_mask.sum() > 20: # Ensure enough samples to fit
            # Train Logistic Regression strictly on OTHER subjects (OOF)
            # Use high regularization (C=0.1) to prevent overfitting
            clf = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
            clf.fit(logits[other_mask].reshape(-1, 1), y_true[other_mask])

            # Predict for current subject
            calibrated_p[subj_mask] = clf.predict_proba(logits[subj_mask].reshape(-1, 1))[:, 1]
        else:
            # Fallback to temperature scaling
            calibrated_p[subj_mask] = 1 / (1 + np.exp(-logits[subj_mask] / global_T))

    return np.clip(calibrated_p, eps, 1 - eps)

def calibrate_beta_global(y_true, p_raw):
    eps = 1e-15
    p_raw = np.clip(p_raw, eps, 1 - eps)
    logits = np.log(p_raw / (1 - p_raw))

    clf = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
    clf.fit(logits.reshape(-1, 1), y_true)

    return clf

def run_meta_stacking_v28(submission_dir='submissions'):
    logger.info("Initializing Ultimate SOTA-v28 Stacking & Beta Calibration Pipeline...")

    # 1. Load labels
    data_loader = DataLoader(data_dir="data")
    metrics_df = data_loader.load_metrics()
    if metrics_df is None:
        logger.error("Could not load metrics file for true labels.")
        return

    targets = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]
    id_col = "subject_id" if "subject_id" in metrics_df.columns else "user_id"

    if 'lifelog_date' in metrics_df.columns:
        metrics_df['date'] = pd.to_datetime(metrics_df['lifelog_date']).dt.date
    elif 'date' not in metrics_df.columns and 'sleep_date' in metrics_df.columns:
        metrics_df['date'] = pd.to_datetime(metrics_df['sleep_date']).dt.date

    metrics_df['date'] = pd.to_datetime(metrics_df['date']).dt.date
    metrics_df = metrics_df.sort_values(by=[id_col, 'date']).reset_index(drop=True)

    # 2. Get OOF files (added 'lr')
    clean_model_names = ['catboost', 'lgbm', 'xgb', 'et', 'rf', 'dl', 'lr']
    oof_files = []
    for m in clean_model_names:
        path = os.path.join(submission_dir, f'oof_{m}_v10.csv')
        if os.path.exists(path):
            oof_files.append(path)

    if len(oof_files) < 2:
        logger.warning(f"Need at least 2 clean OOF files. Found {len(oof_files)}.")
        return

    logger.info(f"Loading {len(oof_files)} clean base models: {[os.path.basename(f) for f in oof_files]}...")
    raw_oof_preds = []
    sub_files = []

    for oof_f in oof_files:
        df = pd.read_csv(oof_f)
        df['date'] = pd.to_datetime(df['date']).dt.date
        df = df.sort_values(by=[id_col, 'date']).reset_index(drop=True)

        if not (df[id_col] == metrics_df[id_col]).all() or not (df['date'] == metrics_df['date']).all():
            logger.error(f"Mismatch in {oof_f}")
            continue

        sub_name = os.path.basename(oof_f).replace('oof_', 'submission_')
        sub_path = os.path.join(submission_dir, sub_name)
        if not os.path.exists(sub_path):
            continue

        raw_oof_preds.append(df[targets].values)
        sub_files.append(sub_path)

    num_models = len(raw_oof_preds)
    y_true = metrics_df[targets].values
    valid_mask = ~np.isnan(y_true)

    sub_dfs = [pd.read_csv(f) for f in sub_files]

    # Pre-Calibrate OOF and Test using out-of-fold subject-wise Beta Calibration
    calibrated_oof_preds = [p.copy() for p in raw_oof_preds]
    calibrated_sub_dfs = [df.copy() for df in sub_dfs]

    logger.info("Applying out-of-fold Subject-Wise Beta Calibration...")
    for m_idx in range(num_models):
        for t_idx, target in enumerate(targets):
            mask = valid_mask[:, t_idx]
            raw_p_oof = raw_oof_preds[m_idx][:, t_idx]
            pred_mask = ~np.isnan(raw_p_oof)
            combined_mask = mask & pred_mask

            if combined_mask.sum() == 0:
                continue

            y_calib = y_true[combined_mask, t_idx]
            p_calib_raw = raw_p_oof[combined_mask]
            subjects = metrics_df.loc[combined_mask, id_col].values

            # Subject-wise beta calibration out-of-fold
            calib_p_oof = raw_p_oof.copy()
            calib_p_oof[combined_mask] = calibrate_beta_subject_wise(y_calib, p_calib_raw, subjects)
            calibrated_oof_preds[m_idx][:, t_idx] = calib_p_oof

            # Global calibration model for Test set (since test set is unseen future)
            clf_global = calibrate_beta_global(y_calib, p_calib_raw)
            raw_p_test = sub_dfs[m_idx][target].values
            eps = 1e-15
            raw_p_test = np.clip(raw_p_test, eps, 1 - eps)
            logits_test = np.log(raw_p_test / (1 - raw_p_test))
            calibrated_sub_dfs[m_idx][target] = clf_global.predict_proba(logits_test.reshape(-1, 1))[:, 1]

    # Output DataFrames for Meta-Stacking
    sample_sub_df = data_loader.load_sample_submission()
    stack_sub_df = sample_sub_df.copy()
    for t in targets:
        stack_sub_df[t] = 0.5

    blended_oof_overall = []
    stacked_oof_overall = []

    logger.info("\n========== Running Stacking vs Blending Target Sweep ==========")

    for t_idx, target in enumerate(targets):
        mask = valid_mask[:, t_idx]
        if mask.sum() == 0:
            continue

        pred_mask = np.ones_like(mask)
        for p in calibrated_oof_preds:
            pred_mask = pred_mask & ~np.isnan(p[:, t_idx])

        combined_mask = mask & pred_mask
        if combined_mask.sum() == 0:
            continue

        # Prepare inputs: columns are base model calibrated predictions
        X_oof = np.column_stack([p[combined_mask, t_idx] for p in calibrated_oof_preds])
        y_oof = y_true[combined_mask, t_idx]

        # Prepare Test inputs
        X_test = np.column_stack([df[target].values for df in calibrated_sub_dfs])

        # 1. SLSQP Blending baseline
        def blend_objective(weights):
            blended = np.dot(X_oof, weights)
            blended = np.clip(blended, 1e-15, 1 - 1e-15)
            return log_loss(y_oof, blended)

        initial_weights = [1.0 / num_models] * num_models
        bounds = [(0.0, 1.0)] * num_models
        constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
        res = minimize(blend_objective, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
        opt_w = res.x / np.sum(res.x)
        blend_score = res.fun
        base_blended = np.clip(np.dot(X_oof, opt_w), 1e-15, 1 - 1e-15)

        # 2. Logistic Regression Meta-Learner Stacking (L2 regularization)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        meta_oof = np.zeros(len(y_oof))
        meta_test_preds = np.zeros(len(X_test))

        best_c = 1.0
        best_meta_loss = 999.0

        for c_val in [0.01, 0.1, 1.0, 10.0, 100.0]:
            fold_scores = []
            for train_idx, val_idx in skf.split(X_oof, y_oof):
                meta_clf = LogisticRegression(C=c_val, max_iter=1000, random_state=42)
                meta_clf.fit(X_oof[train_idx], y_oof[train_idx])
                fold_p = meta_clf.predict_proba(X_oof[val_idx])[:, 1]
                fold_scores.append(log_loss(y_oof[val_idx], fold_p))
            mean_c_loss = np.mean(fold_scores)
            if mean_c_loss < best_meta_loss:
                best_meta_loss = mean_c_loss
                best_c = c_val

        # Generate OOF with best_c
        for train_idx, val_idx in skf.split(X_oof, y_oof):
            meta_clf = LogisticRegression(C=best_c, max_iter=1000, random_state=42)
            meta_clf.fit(X_oof[train_idx], y_oof[train_idx])
            meta_oof[val_idx] = meta_clf.predict_proba(X_oof[val_idx])[:, 1]
            meta_test_preds += meta_clf.predict_proba(X_test)[:, 1] / 5.0

        stack_score = log_loss(y_oof, meta_oof)

        # Select best base prediction path (Stacking vs Blending)
        if stack_score < blend_score:
            logger.info(f"Target {target} -> Stacking chosen. (Stack CV: {stack_score:.5f} vs Blend CV: {blend_score:.5f})")
            base_oof_preds = meta_oof
            base_test_preds = np.clip(meta_test_preds, 1e-15, 1 - 1e-15)
        else:
            logger.info(f"Target {target} -> Blending chosen. (Blend CV: {blend_score:.5f} vs Stack CV: {stack_score:.5f})")
            base_oof_preds = base_blended
            base_test_preds = np.clip(np.dot(X_test, opt_w), 1e-15, 1 - 1e-15)

        # 3. Post-Processing: Out-of-fold Personalized Baseline Calibration
        subjects = metrics_df.loc[combined_mask, id_col].values
        test_subjects = sub_dfs[0][id_col].values

        calibrated_oof = np.zeros_like(base_oof_preds)

        # We search alpha strictly out-of-fold
        for train_idx, val_idx in skf.split(X_oof, y_oof.astype(int)):
            train_subjects = subjects[train_idx]
            train_y = y_oof[train_idx]

            # Subject means strictly on train fold
            subject_means = {}
            global_mean = train_y.mean()

            for s in np.unique(subjects):
                s_mask = train_subjects == s
                if s_mask.sum() > 0:
                    subject_means[s] = (train_y[s_mask].sum() + 5.0 * global_mean) / (s_mask.sum() + 5.0)
                else:
                    subject_means[s] = global_mean

            # Find best alpha
            best_alpha = 1.0
            best_fold_loss = 999.0
            train_preds = base_oof_preds[train_idx]
            train_subj_means = np.array([subject_means[s] for s in train_subjects])

            for alpha in np.linspace(0.5, 1.0, 51):
                p_blend = alpha * train_preds + (1 - alpha) * train_subj_means
                loss = log_loss(train_y, np.clip(p_blend, 1e-15, 1 - 1e-15))
                if loss < best_fold_loss:
                    best_fold_loss = loss
                    best_alpha = alpha

            # Apply to val fold
            val_preds = base_oof_preds[val_idx]
            val_subjects = subjects[val_idx]
            val_subj_means = np.array([subject_means[s] for s in val_subjects])

            calibrated_oof[val_idx] = best_alpha * val_preds + (1 - best_alpha) * val_subj_means

        # Overall best alpha for Test set projection
        global_mean = y_oof.mean()
        subject_means_all = {}
        for s in np.unique(subjects):
            s_mask = subjects == s
            if s_mask.sum() > 0:
                subject_means_all[s] = (y_oof[s_mask].sum() + 5.0 * global_mean) / (s_mask.sum() + 5.0)
            else:
                subject_means_all[s] = global_mean

        best_alpha_global = 1.0
        best_global_loss = 999.0
        subj_means_all = np.array([subject_means_all[s] for s in subjects])

        for alpha in np.linspace(0.5, 1.0, 51):
            p_blend = alpha * base_oof_preds + (1 - alpha) * subj_means_all
            loss = log_loss(y_oof, np.clip(p_blend, 1e-15, 1 - 1e-15))
            if loss < best_global_loss:
                best_global_loss = loss
                best_alpha_global = alpha

        # Project test set using best global alpha
        test_subj_means = np.array([subject_means_all.get(s, global_mean) for s in test_subjects])
        final_test_preds = best_alpha_global * base_test_preds + (1 - best_alpha_global) * test_subj_means

        # Save results
        final_target_loss = log_loss(y_oof, np.clip(calibrated_oof, 1e-15, 1 - 1e-15))
        logger.info(f"Target {target} -> Final Personalized CV: {final_target_loss:.5f} (Alpha: {best_alpha_global:.2f})")

        stack_sub_df[target] = np.clip(final_test_preds, 1e-15, 1 - 1e-15)
        blended_oof_overall.append(min(blend_score, stack_score))
        stacked_oof_overall.append(final_target_loss)

    final_blend_cv = np.mean(blended_oof_overall)
    final_stack_cv = np.mean(stacked_oof_overall)

    logger.info("\n==========================================")
    logger.info("🏁 Ultimate SOTA-v28 Stacking Completed!")
    logger.info(f"  - Clean Base Ensemble CV: {final_blend_cv:.5f}")
    logger.info(f"  - Personalized Calibrated CV: {final_stack_cv:.5f}")
    logger.info("==========================================")

    hybrid_path = os.path.join(submission_dir, 'submission_calibrated_ensemble.csv')
    stack_sub_df.to_csv(hybrid_path, index=False)
    logger.info(f"✅ Best Hybrid Blending/Stacking output written to {hybrid_path}")

if __name__ == "__main__":
    run_meta_stacking_v28()


## 🎯 Part 6: 로컬 OOF 무결성 검증 및 비판적 스코어 분석 (제출 우회)
본 파트에서는 생성된 테스트 예측치들의 포맷 및 값 범위를 엄격하게 자가 진단(Self-diagnosis)하여 로컬 성능을 최종 평가합니다.
**제출 한도 소진으로 인한 전체 파이프라인의 프리징과 과적합 방지를 위해, 리더보드 API 제출은 무조건 집행하지 않고 사용자가 명시적으로 원할 때만 아래의 [선택적 릴리즈 옵션 셀]을 기동하여 제출하도록 강제성을 배제하였습니다.**

In [ ]:
# 1. 로컬 테스트 예측 프레임 로드 및 유효성 엄격 자가 진단
sub_calib_path = 'submissions/submission_calibrated_ensemble.csv'
if os.path.exists(sub_calib_path):
    sub_df = pd.read_csv(sub_calib_path)
    sample_sub = pd.read_csv('data/raw/ch2026_submission_sample.csv')

    print("=== 로컬 예측 파일 자가 검증 결과 ===")
    print("1. 행 개수 일치 여부:", len(sub_df) == len(sample_sub), f"({len(sub_df)} 행)")
    print("2. 누락 값 존재 여부:", sub_df.isnull().sum().sum() == 0)
    print("3. 데이터 값의 [0, 1] 범위 무결성:", (sub_df.iloc[:, 3:] >= 0.0).all().all() and (sub_df.iloc[:, 3:] <= 1.0).all().all())
    print("\n=== 상위 5개 행 미리보기 ===")
    print(sub_df.head())
else:
    print("자가 검증 실패: calibrated ensemble 파일이 아직 생성되지 않았습니다.")

### 🚀 [선택적 옵션] DACON API 리더보드 제출 실행
로컬 테스트에서 교차 검증 스코어 개선이 명확하게 확인되고, 하루 제출 제한 횟수가 남아있는 경우에만 아래 셀을 기동하여 최종 제출을 안전하게 진행합니다.

In [ ]:
# 필요 시 주석을 해제하고 실행하여 제출합니다.
# !.venv/bin/python scripts/submit_calibrated.py